# data conversion scripts

converts everything to zarr files

performs hotpixel correction too

# SPAD512 single bin

In [ ]:
"""
Processes quanta SPAD512 data into a Zarr format. Saves events as normalized points in
space. Also performs hot pixel correction.
"""
%load_ext autoreload
%autoreload 2
%matplotlib inline
from pathlib import Path

from uwb3d import ihpp_data, utils

nframes = 100_000
fps = 100_000
T_exp = nframes / fps
data_dir = Path("data/quanta")
binpath = data_dir / "lift_jump.bin"
nframes = int(fps * T_exp)
quantadata_path = data_dir / f"{binpath.stem}.zarr"
print(f"Processing raw binary file {binpath} into binframes...")
quantaframes = ihpp_data.read_spadbin(
    binpath, nframes=nframes, h=512, w=512, keep_prob=1.0, hotpix_thresh=None, 
)
quantaframes = ihpp_data.SPADHotpixelTool(quantaframes).correct_hotpixels()
ihpp_data.write_quantaframes_zarr(quantadata_path, quantaframes, T_exp=T_exp, save_coords=True)

# SPAD512 raw data folder

In [ ]:
"""
SPAD512 raw data folder (a bunch of .bin files instead of a single one) to Zarr format.
"""
%load_ext autoreload
%autoreload 2
%matplotlib inline
from pathlib import Path

import numpy as np

from uwb3d import ihpp_data
from uwb3d.config import DATA_DIR

datanames = ["opticalsetupsfm-prettydark", "opticalsetupsfm-slightlydark"]
for dataname in datanames:
    print(f"Processing {dataname}...")
    dir_path = DATA_DIR / f"quanta/{dataname}"
    frames = ihpp_data.read_quanta_dir(dir_path, H=512, W=512)

    out_path = dir_path.parent / f"{dataname}.zarr"

    print("Correcting hotpixels...")
    frames = ihpp_data.SPADHotpixelTool(frames).correct_hotpixels()
    ihpp_data.write_quantaframes_zarr(out_path, frames, fps=100_000)
    del frames

# npy file from the wision data

In [ ]:
"""
npy file from the wision data or something to Zarr format.
"""
%load_ext autoreload
%autoreload 2
%matplotlib inline
from pathlib import Path
import numpy as np

from uwb3d import ihpp_data, utils
from uwb3d.config import DATA_DIR

dataname = "lighter"
data_path = DATA_DIR / f"quanta/{dataname}.npy"
a = np.load(data_path)
A = np.unpackbits(a, axis=2)[:, ::-1, ::-1]

out_path = data_path.parent / f"{dataname}.zarr"

A_sliced = A[30000:96000, :, 16:]
A_sliced = ihpp_data.SPADHotpixelTool(A_sliced).correct_hotpixels()
ihpp_data.write_quantaframes_zarr(out_path, A_sliced, fps=1 / 1.05e-5)

# Asynchronous SPAD data conversion

In [ ]:
"""
Asynchronous SPAD data conversion to Zarr format.
"""
%load_ext autoreload
%autoreload 2
%matplotlib inline
import numpy as np
from pathlib import Path

from uwb3d import ihpp_data, utils

T_exp = 0.1
W = 138
H = 91

datapath = Path("data/async-spad/AC_fan/G")
if not datapath.is_dir():
    raise ValueError()
outpath = datapath.parent / f"{datapath.name}.zarr"

points, pixel_timeseries = ihpp_data.read_spadfolder(datapath, h=H, w=W, T_exp=T_exp, normalize=False)

# Make arrays for index structure
offsets = np.zeros((H, W), dtype="int64")
lengths = np.zeros((H, W), dtype="int64")

# Build flat list
flat_list = []
cursor = 0

pixel_counts = np.zeros((H, W), dtype="int64")
for i in range(H):
    if len(pixel_timeseries[i]) != W:
        raise ValueError("All rows must have the same length (W).")
    for j in range(W):
        arr = pixel_timeseries[i][j]
        arr = np.asarray(arr)
        
        offsets[i, j] = cursor
        lengths[i, j] = arr.size
        
        flat_list.append(arr)
        cursor += arr.size
        pixel_counts[i, j] = arr.size
pixel_photo_rates = pixel_counts / T_exp
# Concatenate timestamps
timestamps = np.concatenate(flat_list) if cursor > 0 else np.array([], dtype=float)

data = {
    "timestamps": timestamps,
    "offsets": offsets,
    "lengths": lengths,
    "t": points[:, 0],
    "y": points[:, 1],
    "x": points[:, 2],
}
metadata = {
    "T_exp": T_exp,
    "H": H,
    "W": W,
    "shape": "(H, W)",
    "npoints": points.shape[0],
    "avg_pts_persec_perpixel": np.mean(pixel_photo_rates),
    "max_pts_persec_perpixel": np.max(pixel_photo_rates),
    "min_pts_persec_perpixel": np.min(pixel_photo_rates),
    "med_pts_persec_perpixel": np.median(pixel_photo_rates),
    "stddev_pts_persec_perpixel": np.std(pixel_photo_rates),
}
utils.save_arrs_to_zarr(
    data, outpath,
    chunks={
        "timestamps": (100_000_000,),
        "t": (100_000_000,),
        "y": (100_000_000,),
        "x": (100_000_000,),
    },
    n_workers=48,
    overwrite=True,
    attrs=metadata
)

# Re-encode existing zarr files

In [ ]:
"""
Re-save existing quanta spad files to fix compression or add metadata.
"""
%load_ext autoreload
%autoreload 2
%matplotlib inline
import numpy as np
from pathlib import Path
import shutil

from uwb3d import ihpp_data, utils

folder = Path("data/quanta")
zarrpaths = list(folder.glob("*.zarr"))
skip = [
    "plasma_ball_5_med", "guitar", "blender_bright",
    "cpufan_restarget", "balloon_gun1", "plasma-ball",
    "elevator", "lasagna1", "balloon2", "fan", "balloon_fly1",
    "balloon3", "fountain", "ultrasound_bubble24"
]
include = [
    "Monkey", "Resolution_target_drill"
]
correct_hotpixels = True
for zarrpath in zarrpaths:
    if zarrpath.stem in skip:
        print(f"Skipping {zarrpath}...")
        continue
    if zarrpath.stem not in include:
        print(f"Skipping {zarrpath} (not in include list)...")
        continue
    print(f"Re-saving {zarrpath}...")
    frames, _, _ = ihpp_data.read_quanta_zarr(zarrpath)
    if correct_hotpixels:
        print("Correcting hotpixels...")
        frames = ihpp_data.SPADHotpixelTool(frames).correct_hotpixels()
    # save to temp path first to avoid overwriting if something goes wrong
    zarrpath_tmp = zarrpath.with_suffix(".tmp.zarr")
    ihpp_data.write_quantaframes_zarr(zarrpath_tmp, frames, fps=100_000)
    # delete old zarr
    shutil.rmtree(zarrpath)
    # rename new zarr to old name
    zarrpath_tmp.rename(zarrpath)